# Generic Python on HPC with `python-s3`

This notebook runs a plain Python computation on a TACC Stampede3 compute node using the general-purpose [`python-s3`](https://designsafe-ci.github.io/ds-workflows/apps/python) app — no dedicated Tapis app required.

The example estimates π by Monte Carlo sampling across **all 48 cores of one SKX node** with `concurrent.futures` — no MPI, no pip installs.

Every `python-s3` job runs a staged lifecycle and writes a machine-readable run record (`job-summary.json`):

```
setup (unzip, modules, pip)  →  PRE_SCRIPT  →  main (BINARY, default python3)  →  POST_SCRIPT
```

You can also run this app from the portal: [python-s3 workspace](https://www.designsafe-ci.org/workspace/python-s3?appVersion=1.0.0).

## Install DesignSafe API (dapi)

In [ ]:
%pip install dapi --quiet

In [ ]:
# Import required modules
import json
from pathlib import Path

from dapi import DSClient

In [ ]:
# Initialize DesignSafe client
ds = DSClient()

## Create the input script

`python-s3` needs an input directory containing the main script. On DesignSafe JupyterHub, My Data is mounted at `~/MyData`, so we can write the script directly.

In [ ]:
input_dir = Path.home() / "MyData" / "dapi-examples" / "pi-demo"
input_dir.mkdir(parents=True, exist_ok=True)

pi_script = '''\
"""Monte Carlo estimate of pi across all cores of one node."""
import os
import random
import sys
from concurrent.futures import ProcessPoolExecutor


def count_hits(n: int) -> int:
    rng = random.Random(os.getpid())
    return sum(rng.random() ** 2 + rng.random() ** 2 <= 1.0 for _ in range(n))


if __name__ == "__main__":
    samples = int(sys.argv[1]) if len(sys.argv) > 1 else 10_000_000
    workers = len(os.sched_getaffinity(0))
    chunk = samples // workers
    with ProcessPoolExecutor(workers) as pool:
        hits = sum(pool.map(count_hits, [chunk] * workers))
    total = chunk * workers
    print(f"pi ~= {4 * hits / total:.6f}  ({workers} workers, {total:,} samples)")
'''

(input_dir / "pi.py").write_text(pi_script)
print(f"Wrote {input_dir / 'pi.py'}")

## Job configuration

In [ ]:
# Job configuration parameters
ds_path: str = "/MyData/dapi-examples/pi-demo"  # Path to input files
input_filename: str = "pi.py"  # Main input script filename
max_job_minutes: int = 10  # Maximum runtime in minutes
tacc_allocation: str = (
    "DS-Portal-SPARC2026"  # TACC allocation to charge — change to yours
)
app_id_to_use: str = "python-s3"  # General-purpose Python application ID

In [ ]:
# Convert DesignSafe path to Tapis URI format
input_uri = ds.files.to_uri(ds_path)
print(f"Input Directory Tapis URI: {input_uri}")

In [ ]:
# Generate job request dictionary using app defaults
job_dict = ds.jobs.generate(
    app_id=app_id_to_use,
    input_dir_uri=input_uri,
    script_filename=input_filename,
    max_minutes=max_job_minutes,
    allocation=tacc_allocation,
    queue="skx-dev",  # development queue: fast turnaround for short runs
    archive_system="designsafe",
    archive_path="python-s3-results",
    job_name="mc-pi",
    description="Monte Carlo pi on one Stampede3 node",
    tags=["demo"],
)
print(json.dumps(job_dict, indent=2, default=str))

### Optional customization

The app is configured entirely through the Tapis job request — arguments, a different executable, modules, pip installs, and pre/post scripts. See the [app documentation](https://designsafe-ci.github.io/ds-workflows/apps/python) for all options.

In [ ]:
# Optional: pass command-line arguments to the script (more samples)
# job_dict["parameterSet"]["appArgs"].append({"name": "Arguments", "arg": "50000000"})

# Optional: run something other than Python, e.g. OpenSees-MP (Tcl) on 2 nodes
# job_dict["nodeCount"] = 2
# job_dict["coresPerNode"] = 48
# job_dict["parameterSet"]["envVariables"] = [
#     {"key": "BINARY", "value": "OpenSeesMP"},
#     {"key": "EXTRA_MODULES", "value": "opensees,hdf5/1.14.4"},
#     {"key": "USE_MPI", "value": "True"},
# ]

# Optional: many small input files? Ship ONE zip instead — Tapis stages each
# file as its own transfer (~40s/file under load). The app expands it before
# anything else runs, so even the pre-script can live inside the bundle.
# job_dict["parameterSet"]["envVariables"] = [
#     {"key": "UNZIP_INPUTS", "value": "inputs"},   # inputs.zip in the Input Directory
# ]

## Submit and monitor

In [ ]:
# Submit the job to TACC
submitted_job = ds.jobs.submit(job_dict)
print(f"Job UUID: {submitted_job.uuid}")

In [ ]:
# Monitor job execution until completion.
# timeout_minutes bounds the *monitoring*, not the job — it defaults to the
# job's max_minutes, which queue/staging waits can exhaust, so give it slack.
final_status = submitted_job.monitor(interval=15, timeout_minutes=60)
print(f"Job {submitted_job.uuid} finished with status: {final_status}")

In [ ]:
# Interpret job outcome and display runtime summary
ds.jobs.interpret_status(final_status, submitted_job.uuid)
submitted_job.print_runtime_summary(verbose=False)

## Results

`tapisjob.out` holds the script's stdout — the π estimate — and `job-summary.json` is the app's machine-readable run record: per-stage exit codes and timings, the exact command, the resolved binary, and the loaded modules.

In [ ]:
# Display job output from stdout
stdout_content = submitted_job.get_output_content("tapisjob.out", max_lines=30)
if stdout_content:
    print(stdout_content)

In [ ]:
# Display the machine-readable run record
summary_content = submitted_job.get_output_content("job-summary.json")
if summary_content:
    print(summary_content)

In [ ]:
# List contents of job archive directory
archive_uri = submitted_job.archive_uri
print(f"Archive URI: {archive_uri}")
for item in ds.files.list(archive_uri):
    print(f"- {item.name} ({item.type})")

## Next steps

- [Python App documentation](https://designsafe-ci.github.io/ds-workflows/apps/python) — all inputs, environment variables, pre/post-script recipes
- [Job Resources](https://designsafe-ci.github.io/ds-workflows/guide/job-resources) — choosing nodes, cores, and walltime
- [Parameter Sweeps](https://designsafe-ci.github.io/ds-workflows/guide/parameter-sweeps) — many small runs in one allocation with PyLauncher